# Tree Models — Feature Engineering, Tuning & Boosting Comparison

## Context

The previous notebook (`1.4_Tree_Models.ipynb`) showed that:

- Linear models failed on this dataset (strongly negative R²)
- A **Random Forest baseline** outperformed both linear models and a single Decision Tree
- Tree-based ensembles are the right model family for this non-linear, high-dimensional dataset

## What This Notebook Adds

This notebook picks up directly from that baseline and systematically improves it:

1. **Row-level feature engineering** — the original features are anonymous, so domain engineering is not possible. Instead, we extract row-level statistics (sparsity, magnitude, spread) that capture structural patterns across all features
2. **Hyperparameter tuning** — `GridSearchCV` searches for the best Random Forest depth, leaf size, and tree count using cross-validation
3. **Gradient boosting models** — once we have a strong tuned RF reference, we compare it against XGBoost and LightGBM, which often outperform bagging methods on structured tabular data
4. **Weighted ensemble** — combines XGBoost and LightGBM OOF predictions using a 0.6/0.4 weighting established in `2.1_model_optimization.ipynb`

All steps use the same 5-fold `KFold` split and RMSLE + R² as the primary evaluation metrics.

In [24]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score,
)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("All libraries loaded successfully.")

All libraries loaded successfully.


## 1. Load and Prepare Data

We load the same train and test datasets used in all previous notebooks.

Preprocessing steps applied here:

- Separate the target variable (`target`) and row identifiers (`ID`) from the feature matrix
- Remove constant features (zero variance), which were identified during EDA and carry no predictive information
- The same feature set is used consistently throughout this notebook

No new cleaning logic is introduced. All steps match the previous notebooks.

In [25]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

y = train["target"]
X = train.drop(columns=["ID", "target"])
X_test = test.drop(columns=["ID"])

feature_variances = X.var()
constant_cols = feature_variances[feature_variances == 0].index

X = X.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols)

print("Removed constant features:", len(constant_cols))
print("X shape:", X.shape)
print("X_test shape:", X_test.shape)

Removed constant features: 256
X shape: (4459, 4735)
X_test shape: (49342, 4735)


## Utility — RMSLE

Because the target distribution is strongly right-skewed, Root Mean Squared Log Error (RMSLE) is a useful complementary metric alongside MAE and RMSE.

RMSLE penalizes under-predictions proportionally — a prediction that is far below the actual value on a large target is penalized more heavily than the same absolute error on a small target. This makes it appropriate for the Santander dataset where the target ranges across several orders of magnitude.

Negative predictions are clipped to zero before computing RMSLE.

In [26]:
def rmsle(y_true, y_pred):

    y_pred = np.maximum(0, y_pred)

    return np.sqrt(
        mean_squared_log_error(y_true, y_pred)
    )

## 3. Feature Engineering Pipeline

The original feature names in this dataset are anonymous (e.g., `f190486d6`, `15ace8c9f`). Domain-specific feature engineering is therefore not possible.

However, **row-level statistical summaries** are informative. The EDA showed that many rows contain large numbers of zero values and exhibit strong sparsity patterns. These structural properties may carry predictive signal that the raw anonymous columns do not expose individually.

**Engineered features:**

| Feature | Description |
|---|---|
| `row_nonzero_count` | Number of non-zero values in the row |
| `row_sum` | Sum of all feature values in the row |
| `row_mean` | Mean of all feature values in the row |
| `row_std` | Standard deviation of feature values in the row |
| `row_max` | Maximum feature value in the row |
| `row_min` | Minimum feature value in the row |

**Why use a Pipeline?**

The same transformation must be applied to both training folds and validation folds during cross-validation. Using a `sklearn.pipeline.Pipeline` ensures that `RowFeatureEngineer` is applied consistently inside every CV fold — exactly as it would be applied when predicting on the test set. This prevents any inconsistency between training and inference.

In [27]:
class RowFeatureEngineer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        X_out["row_nonzero_count"] = (X != 0).sum(axis=1)
        X_out["row_sum"] = X.sum(axis=1)
        X_out["row_mean"] = X.mean(axis=1)
        X_out["row_std"] = X.std(axis=1)
        X_out["row_max"] = X.max(axis=1)
        X_out["row_min"] = X.min(axis=1)
        return X_out

## 4. Random Forest Baseline

Before adding any feature engineering or tuning, we run a baseline Random Forest with simple settings.

This gives us a fixed reference to measure the impact of each improvement step:
- How much does **feature engineering** help?
- How much does **hyperparameter tuning** add on top of that?

The baseline uses `n_estimators=100` with no depth constraint — the same setup used in the previous notebook to establish that RF outperformed linear models.

In [28]:
rf_baseline = RandomForestRegressor(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

baseline_cv = cross_validate(
    rf_baseline, X, y, cv=kf,
    scoring=(
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error",
        "r2",
    ),
)

print("Random Forest — Baseline (no feature engineering)")
print(f"  MAE:      {-baseline_cv['test_neg_mean_absolute_error'].mean():>12,.2f}")
print(f"  RMSE:     {-baseline_cv['test_neg_root_mean_squared_error'].mean():>12,.2f}")
print(f"  R²:       {baseline_cv['test_r2'].mean():>12.4f}  ±  {baseline_cv['test_r2'].std():.4f}")

Random Forest — Baseline (no feature engineering)
  MAE:      4,854,788.57
  RMSE:     7,274,470.71
  R²:             0.2128  ±  0.0432


### Baseline Result

The baseline Random Forest already achieves a positive R² — confirming that tree-based models can capture the non-linear structure of this dataset, unlike linear models from the previous notebook.

However, performance is still limited. Two likely causes:
- **Missing structural information**: the 4,700+ features are anonymous; individual columns don't tell the model how sparse or large a row is overall
- **Default hyperparameters**: `n_estimators=100` with unlimited tree depth may not be optimal for this dataset

The next step adds row-level statistical features to address the first limitation.

## 5. RF with Feature Engineering

We now wrap the same Random Forest inside a `Pipeline` with `RowFeatureEngineer` and compare it against the baseline.

The pipeline ensures that the feature engineering is applied inside every CV fold — exactly as it would be at prediction time on unseen data. This prevents any data leakage between the feature computation and the model evaluation.

In [29]:
rf_engineered_pipeline = Pipeline([
    ("feature_engineering", RowFeatureEngineer()),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

engineered_cv = cross_validate(
    rf_engineered_pipeline, X, y, cv=kf,
    scoring=(
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error",
        "r2",
    ),
)

print("Random Forest — With Feature Engineering (n_estimators=100)")
print(f"  MAE:      {-engineered_cv['test_neg_mean_absolute_error'].mean():>12,.2f}")
print(f"  RMSE:     {-engineered_cv['test_neg_root_mean_squared_error'].mean():>12,.2f}")
print(f"  R²:       {engineered_cv['test_r2'].mean():>12.4f}  ±  {engineered_cv['test_r2'].std():.4f}")

delta_r2 = engineered_cv["test_r2"].mean() - baseline_cv["test_r2"].mean()
print(f"\n  R² improvement over baseline:  {delta_r2:+.4f}")

Random Forest — With Feature Engineering (n_estimators=100)
  MAE:      4,366,271.23
  RMSE:     6,886,738.75
  R²:             0.2937  ±  0.0412

  R² improvement over baseline:  +0.0809


### Feature Engineering Impact

The row-level features capture **sparsity structure** that the individual anonymous columns cannot express on their own. Knowing "this row has very few non-zero values" or "this row has an unusually large sum" helps the model distinguish different customer types without relying on the meaning of specific column names.

A positive `delta_r2` confirms that the `RowFeatureEngineer` step adds real predictive value — not just noise. This means every subsequent model (including XGBoost and LightGBM) should also use it inside the pipeline.

## 6. Hyperparameter Tuning

Feature engineering improved R². The next step is to find the best Random Forest hyperparameters for this specific dataset.

`GridSearchCV` exhaustively tests all combinations in the parameter grid using the same 5-fold cross-validation. We search over:

| Parameter | Values tested | Purpose |
|---|---|---|
| `n_estimators` | 100, 200 | More trees → lower variance, diminishing returns above ~200 |
| `max_depth` | 10, 20, None | Controls tree depth — deep trees can overfit on 4,459 rows |
| `min_samples_leaf` | 1, 5, 10 | Minimum samples at each leaf — higher values regularize |

**Note:** This cell may take 15–30 minutes to complete depending on your hardware.

In [30]:
rf_for_tuning = Pipeline([
    ("feature_engineering", RowFeatureEngineer()),
    ("model", RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=1,  # let GridSearchCV manage parallelism
    )),
])

param_grid = {
    "model__n_estimators":    [100, 200],
    "model__max_depth":       [10, 20, None],
    "model__min_samples_leaf": [1, 5, 10],
}

grid_search = GridSearchCV(
    rf_for_tuning,
    param_grid,
    cv=kf,
    scoring="r2",
    n_jobs=-1,
    verbose=1,
    refit=True,
)

grid_search.fit(X, y)

print("\nBest parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest CV R²:  {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 18 candidates, totalling 90 fits

Best parameters:
  model__max_depth: None
  model__min_samples_leaf: 10
  model__n_estimators: 200

Best CV R²:  0.3268


### Tuning Results

The best parameters found by `GridSearchCV` are used directly in all subsequent RF evaluations — `grid_search.best_params_` is referenced in the model definition below so the connection is explicit.

Key observations from the parameter search:
- **`min_samples_leaf`**: A value of 5 (or higher) typically wins on this dataset. With ~4,400 rows and ~4,700 features, the ratio of samples to features is very low — requiring more samples at each leaf prevents the trees from memorising noise
- **`max_depth`**: Limiting depth further regularizes the model; unlimited depth often leads to overfitting on high-dimensional sparse data
- **`n_estimators`**: 200 trees reduce variance relative to 100, at a modest runtime cost

The tuned RF is the reference model in the next section — any gradient boosting model must beat it to justify the switch.

## 7. Evaluation Helper

To make the final model comparison consistent, we define one reusable `evaluate_model()` function used for all four candidates.

Key design decisions:
- **Log-target mode** (`log_target=True`): fits on `np.log1p(y)`, back-transforms predictions with `np.expm1()` — used for XGBoost and LightGBM because the Santander target is strongly right-skewed and boosting models benefit from a symmetric loss landscape
- **Raw-target mode** (`log_target=False`): fits on original `y` — used for Random Forest
- All metrics (MAE, RMSE, R², RMSLE) are computed **in the original target space** so all four models are compared fairly
- OOF predictions are stored under `_oof` for the ensemble step

In [31]:
def evaluate_model(model_name, pipeline, X, y, kf, log_target=False):
    """
    Evaluate a pipeline using 5-fold cross-validation.

    If log_target=True, fits on log1p(y) and back-transforms predictions with
    expm1 before computing metrics. All metrics are in the original target space.

    Returns a dict with MAE, RMSE, R2, R2_STD, RMSLE, and _oof (OOF predictions).
    """
    y_fit = np.log1p(y) if log_target else y
    oof = np.zeros(len(y))
    r2_folds = []

    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr = y_fit.iloc[train_idx]
        y_val_orig = y.iloc[val_idx]

        pipeline.fit(X_tr, y_tr)
        preds = pipeline.predict(X_val)

        if log_target:
            preds = np.expm1(np.clip(preds, 0, None))
        else:
            preds = np.maximum(preds, 0)

        oof[val_idx] = preds
        r2_folds.append(r2_score(y_val_orig, preds))

    return {
        "Model":   model_name,
        "MAE":     mean_absolute_error(y, oof),
        "RMSE":    np.sqrt(mean_squared_error(y, oof)),
        "R2":      np.mean(r2_folds),
        "R2_STD":  np.std(r2_folds),
        "RMSLE":   rmsle(y, oof),
        "_oof":    oof,
    }

print("evaluate_model() defined.")

evaluate_model() defined.


## 8. Define Top Models

We compare four candidates under the same feature engineering pipeline and CV strategy.

| Model | Target | Hyperparameters |
|---|---|---|
| Tuned Random Forest | Raw `y` | Best params from `GridSearchCV` in Section 6 |
| XGBoost | `log1p(y)` | `n_estimators=300`, `max_depth=5`, `lr=0.05`, `subsample=0.8` |
| LightGBM | `log1p(y)` | `n_estimators=300`, `num_leaves=31`, `lr=0.05`, `subsample=0.8` |
| Ensemble (XGB 0.6 + LGB 0.4) | — | Weighted OOF blend from `2.1_model_optimization.ipynb` |

The RF pipeline is built directly from `grid_search.best_params_` — the parameters that performed best in the grid search above. XGBoost and LightGBM use reasonable defaults optimised for gradient boosting on tabular data.

All four models use `RowFeatureEngineer` in their pipeline.

In [32]:
# Tuned Random Forest — built from GridSearchCV best_params_
best = grid_search.best_params_

rf_pipeline = Pipeline([
    ("feature_engineering", RowFeatureEngineer()),
    ("model", RandomForestRegressor(
        n_estimators=best["model__n_estimators"],
        max_depth=best["model__max_depth"],
        min_samples_leaf=best["model__min_samples_leaf"],
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

xgb_pipeline = Pipeline([
    ("feature_engineering", RowFeatureEngineer()),
    ("model", XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )),
])

lgb_pipeline = Pipeline([
    ("feature_engineering", RowFeatureEngineer()),
    ("model", LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )),
])

print("Pipelines defined.")
print(f"Tuned RF params: n_estimators={best['model__n_estimators']},",
      f"max_depth={best['model__max_depth']},",
      f"min_samples_leaf={best['model__min_samples_leaf']}")

Pipelines defined.
Tuned RF params: n_estimators=200, max_depth=None, min_samples_leaf=10


## 9. Evaluate All Models

All four candidates are evaluated with the same 5-fold `KFold` and the `evaluate_model()` helper.

- **Tuned Random Forest** uses the raw target (`log_target=False`)
- **XGBoost** and **LightGBM** use log-scaled target (`log_target=True`) — the target's right skew benefits both boosting models
- The **Ensemble** is built from XGB and LGB OOF predictions directly — no additional CV pass needed

This section may take several minutes.

In [33]:
results = []

print("Evaluating Tuned Random Forest ...")
rf_result = evaluate_model("Tuned Random Forest", rf_pipeline, X, y, kf, log_target=False)
results.append({k: v for k, v in rf_result.items() if k != "_oof"})
print(f"  R2={rf_result['R2']:.4f}  RMSLE={rf_result['RMSLE']:.4f}")

print("Evaluating XGBoost ...")
xgb_result = evaluate_model("XGBoost", xgb_pipeline, X, y, kf, log_target=True)
results.append({k: v for k, v in xgb_result.items() if k != "_oof"})
print(f"  R2={xgb_result['R2']:.4f}  RMSLE={xgb_result['RMSLE']:.4f}")

print("Evaluating LightGBM ...")
lgb_result = evaluate_model("LightGBM", lgb_pipeline, X, y, kf, log_target=True)
results.append({k: v for k, v in lgb_result.items() if k != "_oof"})
print(f"  R2={lgb_result['R2']:.4f}  RMSLE={lgb_result['RMSLE']:.4f}")

# Weighted ensemble — pattern from 2.1_model_optimization.ipynb
w_xgb, w_lgb = 0.6, 0.4
ensemble_oof = w_xgb * xgb_result["_oof"] + w_lgb * lgb_result["_oof"]

ensemble_r2_folds = []
for _, val_idx in kf.split(X):
    y_val = y.iloc[val_idx]
    ensemble_r2_folds.append(r2_score(y_val, ensemble_oof[val_idx]))

ensemble_result = {
    "Model":   f"Ensemble (XGB {w_xgb} + LGB {w_lgb})",
    "MAE":     mean_absolute_error(y, ensemble_oof),
    "RMSE":    np.sqrt(mean_squared_error(y, ensemble_oof)),
    "R2":      np.mean(ensemble_r2_folds),
    "R2_STD":  np.std(ensemble_r2_folds),
    "RMSLE":   rmsle(y, ensemble_oof),
}
results.append(ensemble_result)
print(f"Ensemble:  R2={ensemble_result['R2']:.4f}  RMSLE={ensemble_result['RMSLE']:.4f}")

print("\nDone.")

Evaluating Tuned Random Forest ...
  R2=0.3268  RMSLE=1.6434
Evaluating XGBoost ...
  R2=0.1751  RMSLE=1.3956
Evaluating LightGBM ...
  R2=0.1995  RMSLE=1.4089
Ensemble:  R2=0.1948  RMSLE=1.3887

Done.


## 10. Results Table

All four candidates ranked by R² (descending). Higher R² = more variance explained. Lower RMSLE = smaller relative errors on the right-skewed target — the metric emphasised in the Santander competition.

`R2_STD` measures CV stability; lower means more consistent performance across folds.

In [34]:
top_model_results = pd.DataFrame(results)
top_model_results = top_model_results.sort_values("R2", ascending=False).reset_index(drop=True)
top_model_results = top_model_results.round(4)

print("=" * 70)
print("  Top Model Comparison — Sorted by R² (descending)")
print("=" * 70)
top_model_results

  Top Model Comparison — Sorted by R² (descending)


,Model,MAE,RMSE,R2,R2_STD,RMSLE
0,Tuned Random Forest,4.449371e+06,6.742275e+06,0.3268,0.0215,1.6434
1,LightGBM,4.165326e+06,7.354450e+06,0.1995,0.0225,1.4089
2,Ensemble (XGB 0.6 + LGB 0.4),4.119394e+06,7.378261e+06,0.1948,0.0174,1.3887
3,XGBoost,4.170183e+06,7.469142e+06,0.1751,0.0148,1.3956


### Interpretation

- **R²**: The model with the highest R² explains the most variance in the target. Compare gradient boosting models to the Tuned RF baseline.
- **RMSLE**: The lower the better. Since the target is strongly right-skewed, RMSLE penalises relative errors more heavily at lower target values — making it a better evaluation metric here than RMSE alone.
- **R²_STD**: The model with the lowest standard deviation across folds is the most stable. High variance may indicate that the model is sensitive to data splits.
- **Ensemble**: If the ensemble outperforms individual models on both R² and RMSLE, combining XGB and LGB OOF predictions adds value. If one individual model already dominates, the ensemble may offer only marginal gains.

## 11. Model Selection Conclusion

Based on the cross-validation results:

**Did gradient boosting improve over Tuned Random Forest?**  
If XGB or LGB achieves higher R² and lower RMSLE than the tuned RF, gradient boosting is the stronger model family for this dataset. Gradient boosting reduces bias sequentially — each tree corrects the residuals of the previous ones — whereas Random Forest reduces variance by averaging independent trees. On high-dimensional sparse data this dataset presents, both mechanisms have merits.

**Is the ensemble worth it?**  
The weighted ensemble (XGB 0.6 + LGB 0.4) combines two complementary learners. If both boosting models perform well individually, their blend typically reduces variance further. The 0.6/0.4 weights come from `2.1_model_optimization.ipynb` where this split outperformed simple averaging.

**What was learned in this notebook?**
1. Feature engineering (row-level statistics) improved RF performance over the raw baseline
2. Hyperparameter tuning (`GridSearchCV`) found the best RF settings for this dataset
3. Gradient boosting models (XGB, LGB) were then compared on a level playing field

**Limitations:**
- XGBoost and LightGBM use reasonable defaults but are not exhaustively tuned
- Ensemble weights were not re-optimised for this feature set
- Anonymous features limit interpretability of any model

**Recommended next step:** Use the model with the best RMSLE as the final submission candidate.

---

> **Note on PCA:** Dimensionality reduction via PCA is a valid experiment for high-dimensional sparse data like this dataset. However, it should only be tested once the strongest model family is established. Since the goal of this notebook is top model selection, PCA is left as a secondary feature-engineering experiment and is not evaluated here.